In [1]:
import os
import shutil
import random
from ultralytics import YOLO

# --- CONFIGURATION ---
RAW_DATASET_DIR = "Datasets/face_database"  
OUTPUT_YOLO_DIR = "Datasets/yolo_dataset"    
TRAIN_SPLIT = 0.8 

In [ ]:
# 1. Load a pre-trained detector to find faces/heads automatically
# This ensures we get true coordinates, not just full-image fallbacks
face_locator = YOLO("yolo26n.pt") 

# Clean old dataset paths to avoid overlaps
if os.path.exists(OUTPUT_YOLO_DIR):
    shutil.rmtree(OUTPUT_YOLO_DIR)

for split in ['train', 'val']:
    os.makedirs(os.path.join(OUTPUT_YOLO_DIR, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_YOLO_DIR, split, 'labels'), exist_ok=True)

classes = sorted([d for d in os.listdir(RAW_DATASET_DIR) if os.path.isdir(os.path.join(RAW_DATASET_DIR, d))])
print(f"Generating proper coordinates for classes: {classes}")

for class_id, class_name in enumerate(classes):
    class_path = os.path.join(RAW_DATASET_DIR, class_name)
    images = [f for f in os.listdir(class_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    
    random.shuffle(images)
    split_idx = int(len(images) * TRAIN_SPLIT)
    
    for idx, img_name in enumerate(images):
        src_img_path = os.path.join(class_path, img_name)
        
        # Locate the person/face in the training image
        results = face_locator(src_img_path, verbose=False)
        
        h, w = 0, 0
        x_center, y_center, norm_w, norm_h = 0.5, 0.5, 1.0, 1.0
        found_box = False
        
        for r in results:
            if len(r.boxes) > 0:
                # Grab the highest confidence bounding box found in the image
                box = r.boxes[0] 
                # Get normalized center coordinates (x, y, w, h)
                x_center, y_center, norm_w, norm_h = map(float, box.xywhn[0])
                found_box = True
                break
        
        split = 'train' if idx < split_idx else 'val'
        dest_filename = f"{class_name}_{img_name}"
        
        # Copy image asset over
        shutil.copy(src_img_path, os.path.join(OUTPUT_YOLO_DIR, split, 'images', dest_filename))
        
        # Save real calculated bounding box labels
        label_filename = f"{class_name}_{img_name.rsplit('.', 1)[0]}.txt"
        label_file_path = os.path.join(OUTPUT_YOLO_DIR, split, 'labels', label_filename)
        
        with open(label_file_path, 'w') as f:
            f.write(f"{class_id} {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}\n")

# Rebuild configuration map
yaml_content = f"""
path: {os.path.abspath(OUTPUT_YOLO_DIR)}
train: train/images
val: val/images

nc: {len(classes)}
names: {classes}
"""

# Combine the absolute path and file name safely
file_path = os.path.join(os.path.abspath(OUTPUT_YOLO_DIR), "data.yaml")

with open(file_path, "w") as f:
  f.write(yaml_content.strip())

# with open("data.yaml", "w") as f:
    # f.write(yaml_content.strip())

print("✅ Data successfully rematted with actual target coordinates!")


Generating proper coordinates for classes: ['Animesh', 'Nivan Tyagi', 'Rakesh Ranjan', 'Shrinav Tyagi', 'Sunny Singh', 'Tejrit Tyagi', 'Vimala']
✅ Data successfully rematted with actual target coordinates!


#### **Step 2: Retrain the Custom Model (train.py)**

In [14]:
from ultralytics import YOLO

yml_data_path = os.path.join(os.path.abspath(OUTPUT_YOLO_DIR), "data.yaml")

model = YOLO("yolo26n.pt") 

model.train(
    data=yml_data_path,    
    epochs=40,           # 40 epochs is a good baseline balance for facial datasets
    imgsz=640,           
    device="cpu",        # Change to 0 or 'cuda' if an NVIDIA GPU is available
    project=os.path.abspath(OUTPUT_YOLO_DIR),  # Anchors directory destination to your Datasets folder
    name="train",        
    exist_ok=True        
)
# print("✅ Retraining complete! New smart weights loaded to {os.path.abspath(OUTPUT_YOLO_DIR)}/weights/best.pt")
print(f"✅ Retraining complete! New smart weights loaded to {os.path.abspath(OUTPUT_YOLO_DIR)}/train/weights/best.pt")


Ultralytics 8.4.163 🚀 Python-3.12.3 torch-2.14.0+cu130 CPU (AMD Ryzen 9 7950X3D 16-Core Processor)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/rakesh/GitRepo/Object-Detection-Models/Datasets/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi

In [5]:
os.path.join(os.path.abspath(OUTPUT_YOLO_DIR), 'train', 'weights', "best.pt")
# weights_path = os.path.join(os.path.abspath(OUTPUT_YOLO_DIR), 'train', 'weights', "best.pt")
# print(weights_path)
# os.path.exists(weights_path)

'/home/rakesh/GitRepo/Object-Detection-Models/yolo_dataset/train/weights/best.pt'

In [ ]:
# os.path.abspath('Datsets')
os.path.join(os.path.abspath('Datasets'), OUTPUT_YOLO_DIR, 'train', 'weights', "best.pt")
# weights_path = os.path.join(os.path.abspath('Datasets'), OUTPUT_YOLO_DIR, 'train', 'weights', "best.pt")

'/home/rakesh/GitRepo/Object-Detection-Models/Datasets/yolo_dataset/train/weights/best.pt'

In [2]:
import os
import cv2
from ultralytics import YOLO

# Dynamic path resolution configuration
OUTPUT_YOLO_DIR = "yolo_dataset" 
# weights_path = os.path.join(os.path.abspath(OUTPUT_YOLO_DIR), 'train', 'weights', "best.pt")
weights_path = os.path.join(os.path.abspath('Datasets'), OUTPUT_YOLO_DIR, 'train', 'weights', "best.pt")

# Fallback check to avoid path errors
if not os.path.exists(weights_path):
    # Check alternative common directory structure format
    weights_path = "Datasets/train/weights/best.pt"

print(f"Loading custom network layers from: {weights_path}")

try:
    model = YOLO(weights_path)
    print("Successfully initialized custom facial dataset weights!")
    print(f"Recognized Identity Profiles: {list(model.names.values())}")
except Exception as e:
    print(f"CRITICAL: Failed to load target path: {e}")
    exit()

cap = cv2.VideoCapture(0)
print("\n--- Live Recognition Engine Successfully Initialized ---")

while True:
    ret, frame = cap.read()
    if not ret:
        print("Dropped video stream frame instance.")
        break

    # REMOVED strict classes parameter filter. 
    # Lowered conf to 0.25 to catch emerging patterns while testing.
    results = model(frame, stream=True, conf=0.25)
    
    for r in results:
        boxes = r.boxes
        if len(boxes) == 0:
            continue
            
        for box in boxes:
            x1, y1, x2, y2 = map(int, box.xyxy)
            class_id = int(box.cls)
            
            # Map the class index directly onto your folder identities structure array
            if class_id in model.names:
                person_name = model.names[class_id]
            else:
                # Dynamic fallback catch for baseline tracking indicators
                person_name = f"Target #{class_id}"
                
            confidence = float(box.conf) * 100
            
            # --- Rendering Graphical UI Overlays ---
            # Green rectangle boundaries box frame
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            label_text = f"{person_name} [{confidence:.1f}%]"
            
            # Calculate banner dimensions safely
            (text_w, text_h), _ = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
            cv2.rectangle(frame, (x1, y1 - 25), (x1 + text_w + 10, y1), (0, 255, 0), -1)
            
            # Text string overlay
            cv2.putText(frame, label_text, (x1 + 5, y1 - 7), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    cv2.imshow("Custom Face Recognition System", frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print("System execution terminated cleanly.")

Loading custom network layers from: /home/rakesh/GitRepo/Object-Detection-Models/Datasets/yolo_dataset/train/weights/best.pt
Successfully initialized custom facial dataset weights!
Recognized Identity Profiles: ['Animesh', 'Nivan Tyagi', 'Rakesh Ranjan', 'Shrinav Tyagi', 'Sunny Singh', 'Tejrit Tyagi', 'Vimala']

--- Live Recognition Engine Successfully Initialized ---

0: 480x640 (no detections), 17.6ms
Speed: 0.8ms preprocess, 17.6ms inference, 0.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 16.9ms
Speed: 0.8ms preprocess, 16.9ms inference, 0.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 16.9ms
Speed: 0.5ms preprocess, 16.9ms inference, 0.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 17.7ms
Speed: 0.5ms preprocess, 17.7ms inference, 0.1ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 17.2ms
Speed: 0.4ms preprocess, 17.2ms inference, 0.1ms postprocess 

In [3]:
import os
import cv2
from ultralytics import YOLO

# --- DYNAMIC PATH CONFIGURATION ---
OUTPUT_YOLO_DIR = "yolo_dataset"

# Your specific path string layout construction
weights_path = os.path.join(os.path.abspath('Datasets'), OUTPUT_YOLO_DIR, 'train', 'weights', "best.pt")

# Safety structural fallback check: If the nested directory path above is missing,
# it automatically defaults to the clean directory path output format: Datasets/train/weights/best.pt
if not os.path.exists(weights_path):
    weights_path = os.path.join(os.path.abspath('Datasets'), 'train', 'weights', "best.pt")

print(f"Loading custom network weights layer from: {weights_path}")

try:
    model = YOLO(weights_path)
    print("Successfully initialized custom facial dataset weights!")
    print(f"Tracking strict target index array: {list(model.names.keys())}")
    print(f"Recognized Identity Profiles: {list(model.names.values())}")
except Exception as e:
    print(f"CRITICAL PATH SELECTION ERROR: {e}")
    exit()

# --- LIVE CAMERA FEED CONTINUITY ENGINE ---
cap = cv2.VideoCapture(0)
print("\n--- Live Recognition Engine Successfully Initialized ---")

while True:
    ret, frame = cap.read()
    if not ret:
        print("Dropped video stream frame instance.")
        break

    # Initialize frame tracker backup matrix window
    annotated_frame = frame.copy()

    # Pass frame stream processing cleanly. 
    # Lowered confidence boundary step slightly to bypass full-image box constraints.
    results = model(frame, stream=True, conf=0.25)
    
    for r in results:
        boxes = r.boxes
        if len(boxes) == 0:
            continue
            
        for box in boxes:
            x1, y1, x2, y2 = map(int, box.xyxy)
            class_id = int(box.cls)
            
            # Extract names cleanly from profile dictionary map lists
            person_name = model.names.get(class_id, f"Target #{class_id}")
            confidence = float(box.conf) * 100
            
            # --- Rendering Graphic Interface Design ---
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            label_text = f"{person_name} [{confidence:.1f}%]"
            
            (text_w, text_h), _ = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
            cv2.rectangle(frame, (x1, y1 - 25), (x1 + text_w + 10, y1), (0, 255, 0), -1)
            cv2.putText(frame, label_text, (x1 + 5, y1 - 7), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
            
            # Update background loop pointer
            annotated_frame = frame

    cv2.imshow("Custom Face Recognition System", annotated_frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print("System execution terminated cleanly.")


Loading custom network weights layer from: /home/rakesh/GitRepo/Object-Detection-Models/Datasets/yolo_dataset/train/weights/best.pt
Successfully initialized custom facial dataset weights!
Tracking strict target index array: [0, 1, 2, 3, 4, 5, 6]
Recognized Identity Profiles: ['Animesh', 'Nivan Tyagi', 'Rakesh Ranjan', 'Shrinav Tyagi', 'Sunny Singh', 'Tejrit Tyagi', 'Vimala']

--- Live Recognition Engine Successfully Initialized ---

0: 480x640 (no detections), 17.1ms
Speed: 0.9ms preprocess, 17.1ms inference, 0.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 16.8ms
Speed: 0.8ms preprocess, 16.8ms inference, 0.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 17.6ms
Speed: 0.4ms preprocess, 17.6ms inference, 0.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 17.1ms
Speed: 0.6ms preprocess, 17.1ms inference, 0.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 16.